# ChemBreak Adaptive Jailbreak v1.1

GitHub to Colab Enterprise runner for `https://github.com/Jollychuks/ChemBreak`.

Project folder: `ChemBreak_Adaptive_Jailbreak_v1_1`

Run TEST first. Use PILOT to calibrate and freeze the attack budget, thresholds, model IDs, route-switch limit, verifier behavior, and prompts. Run PRODUCTION only after the methodology is frozen.


## 1. Clone or refresh the ChemBreak GitHub repository

This cell changes only the ephemeral Colab Enterprise runtime. It never pushes to GitHub.


In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH = "main"
PROJECT_SUBDIR = "ChemBreak_Adaptive_Jailbreak_v1_1"

WORK_ROOT = Path("/content") if Path("/content").exists() else (Path.home() / "chembreak_colab")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORK_ROOT / "ChemBreak"

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

PROJECT_DIR = REPO_DIR / PROJECT_SUBDIR
assert PROJECT_DIR.exists(), f"Project folder not found: {PROJECT_DIR}. Upload {PROJECT_SUBDIR} to GitHub first."
os.chdir(PROJECT_DIR)
print("Repository:", REPO_DIR)
print("Project directory:", PROJECT_DIR)
print("Git commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 2. Install dependencies

The package uses the CUDA-enabled PyTorch already present in the Google Cloud runtime. RDKit is installed for deterministic chemistry verification.


In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e .

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


## 3. Configure the run and model roles

`RUN_ID` must stay the same when resuming an interrupted run.

The attacker defaults to the Gemini Pro preview endpoint already proven to work in your earlier ChemBreak cloud run. If your Model Garden console shows a separate Gemini 3.5 Pro Preview ID, replace only `ATTACKER_MODEL` with that exact ID.


In [ ]:
GCP_PROJECT = "rs-foundsecft-mghasemi"
GCS_BUCKET = "rs-foundsecft-mghasemi-default-1"

RUN_MODE = "test"        # test | pilot | production
RUN_ID = "test_001"      # keep stable to resume this run

# Model roles
ATTACKER_MODEL = "gemini-3.1-pro-preview"  # replace only if your console shows an exact Gemini 3.5 Pro Preview ID
SAFETY_JUDGE_MODEL = "openai/gpt-oss-120b-maas"
CHEMISTRY_JUDGE_MODEL = "gemini-2.5-pro"
ADJUDICATOR_MODEL = "meta/llama-4-maverick-17b-128e-instruct-maas"

# Completed V15 production task bank
TASK_BANK_PREFIX = "ChemBreak_V15/outputs/production/"

# Durable checkpoints and restricted transcripts
GCS_BASE_URI = f"gs://{GCS_BUCKET}/ChemBreak_Adaptive_Jailbreak_v1_1"
GCS_OUTPUT_URI = f"{GCS_BASE_URI}/{RUN_MODE}/{RUN_ID}"

print("Project:", GCP_PROJECT)
print("Mode:", RUN_MODE)
print("Run ID:", RUN_ID)
print("Output:", GCS_OUTPUT_URI)
print("Attacker:", ATTACKER_MODEL)
print("Safety judge:", SAFETY_JUDGE_MODEL)
print("Chemistry judge:", CHEMISTRY_JUDGE_MODEL)
print("Adjudicator:", ADJUDICATOR_MODEL)


## 4. Locate the final ChemBreak task bank in Google Cloud Storage


In [ ]:
from google.cloud import storage

client = storage.Client(project=GCP_PROJECT)
blobs = list(client.list_blobs(GCS_BUCKET, prefix=TASK_BANK_PREFIX))
matches = [b for b in blobs if b.name.lower().endswith(".csv") and "final_task_bank" in b.name.lower()]
matches.sort(key=lambda b: b.updated or 0, reverse=True)

if not matches:
    raise FileNotFoundError(
        f"No final_task_bank CSV found under gs://{GCS_BUCKET}/{TASK_BANK_PREFIX}. "
        "Change TASK_BANK_PREFIX or set TASK_BANK_URI manually."
    )

print("Matching final task banks:")
for i, b in enumerate(matches, 1):
    print(f"{i}. gs://{GCS_BUCKET}/{b.name} | updated={b.updated}")

TASK_BANK_URI = f"gs://{GCS_BUCKET}/{matches[0].name}"
print("Selected:", TASK_BANK_URI)


## 5. Create the local runtime configuration

`configs/runtime.yaml` is ignored by Git.


In [ ]:
import subprocess
subprocess.run([
    "python", "scripts/create_runtime_config.py",
    "--template", "configs/gcp.yaml",
    "--output", "configs/runtime.yaml",
    "--project", GCP_PROJECT,
    "--run-mode", RUN_MODE,
    "--run-id", RUN_ID,
    "--task-bank-uri", TASK_BANK_URI,
    "--gcs-output-uri", GCS_OUTPUT_URI,
    "--attacker-model", ATTACKER_MODEL,
    "--safety-judge-model", SAFETY_JUDGE_MODEL,
    "--chemistry-judge-model", CHEMISTRY_JUDGE_MODEL,
    "--adjudicator-model", ADJUDICATOR_MODEL,
], check=True)

CONFIG = "configs/runtime.yaml"
print(Path(CONFIG).read_text()[:7000])


## 6. Preflight

Preflight checks the GPU, task-bank schema, target Hugging Face repositories, RDKit verifier, and harmless connectivity calls to all required attacker/judge roles.

Llama 4 Maverick must be enabled in Model Garden, its EULA accepted, and it is called in `us-east5`.

If a required model is unavailable, preflight stops before any jailbreak execution.


In [ ]:
!python scripts/preflight.py --config $CONFIG


## 7. Prepare frozen attack assets

For every selected task, this generates the repeated single-turn set, fixed multi-turn sequence, and four-route adaptive graph once. The same assets are reused across ChemDFM, ChemLLM, and LlaSMol.


In [ ]:
!python scripts/run.py prepare --config $CONFIG


## 8. Run ChemDFM

Runs C0, C1, C2, and C3. ChemDFM is loaded once and reused across the target block.


In [ ]:
!python scripts/run.py execute --config $CONFIG --target chemdfm


## 9. Run ChemLLM


In [ ]:
!python scripts/run.py execute --config $CONFIG --target chemllm


## 10. Run LlaSMol


In [ ]:
!python scripts/run.py execute --config $CONFIG --target llasmol


## 11. Rebuild aggregate metrics

Metrics include alignment-breach rate, effective-chemical-breach rate, query efficiency, judge disagreement, adjudication rate, and verifier contradiction rate.


In [ ]:
!python scripts/run.py metrics --config $CONFIG


## 12. Resume behavior

If the runtime stops, reconnect to Colab Enterprise, rerun Sections 1 through 6 with the same `RUN_MODE` and `RUN_ID`, then rerun prepare and the target cells. GCS checkpoints are restored and completed task-target-condition units are skipped.
